# Stage 09: HERC Portfolio Construction

This notebook compares `HRP` and `HERC` under the covariance estimators introduced in Stage 8 while retaining the Phase 1 portfolio construction baseline set:

- Equal Weight
- Inverse Volatility
- HRP
- HERC


In [ ]:
from __future__ import annotations

from pathlib import Path
import sys

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import Markdown, display

project_root = Path.cwd().resolve()
if project_root.name == "09_herc_portfolio_construction":
    project_root = project_root.parents[1]

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.clustering import HERCAllocator, compare_hrp_herc_weights
from src.covariance import CovarianceFactory
from src.dashboard.plots import plot_weight_comparison
from src.optimization import EqualWeightAllocator, HRPAllocator, InverseVolatilityAllocator
from src.optimization.hrp_allocator import allocate_hrp_weights
from src.clustering.herc_allocator import allocate_herc_weights, covariance_to_correlation
from src.covariance.distance import compute_distance_matrix
from src.clustering.hierarchical import compute_linkage_matrix


## Load Data

Use deterministic synthetic returns so the notebook is reproducible without external data access.

In [ ]:
rng = np.random.default_rng(2026)
dates = pd.date_range(start="2021-01-01", periods=504, freq="B")
volatility = np.array([0.009, 0.011, 0.014, 0.017, 0.010])
correlation_driver = rng.normal(0.0, 0.006, size=(len(dates), 1))
idiosyncratic = rng.normal(0.0004, volatility, size=(len(dates), 5))
returns_df = pd.DataFrame(
    correlation_driver + idiosyncratic,
    index=dates,
    columns=["Equity", "IT", "Gold", "Bonds", "Energy"],
)
returns_df.head()


## Compute Covariance

Evaluate the four covariance estimators from Phase 2A.1.

In [ ]:
covariance_methods = {
    "sample": {},
    "ledoit_wolf": {},
    "ewma": {"span": 126},
    "ewma_ledoit_wolf": {"span": 126},
}

covariance_results = {
    method: CovarianceFactory.compute(returns_df, method=method, **kwargs)
    for method, kwargs in covariance_methods.items()
}

for method, covariance_matrix in covariance_results.items():
    display(Markdown(f"### {method}"))
    display(covariance_matrix.round(6))


## HRP Allocation

In [ ]:
equal_weight = pd.Series(
    EqualWeightAllocator().optimize(returns_df),
    index=returns_df.columns,
    name="Equal Weight",
)
inverse_vol = pd.Series(
    InverseVolatilityAllocator().optimize(returns_df),
    index=returns_df.columns,
    name="Inverse Volatility",
)

hrp_weights_by_method: dict[str, pd.Series] = {}
for method, covariance_matrix in covariance_results.items():
    correlation_matrix = covariance_to_correlation(covariance_matrix)
    distance_matrix = compute_distance_matrix(correlation_matrix)
    linkage_matrix = compute_linkage_matrix(distance_matrix, method="single")
    hrp_weights_by_method[method] = allocate_hrp_weights(covariance_matrix, linkage_matrix)

pd.DataFrame(hrp_weights_by_method).round(4)


## HERC Allocation

In [ ]:
herc_weights_by_method: dict[str, pd.Series] = {}
for method, kwargs in covariance_methods.items():
    allocator = HERCAllocator(covariance_method=method, covariance_kwargs=kwargs)
    herc_weights_by_method[method] = pd.Series(
        allocator.optimize(returns_df),
        index=returns_df.columns,
    )

pd.DataFrame(herc_weights_by_method).round(4)


## Weight Comparison

Compare HRP and HERC using the same covariance estimate.

In [ ]:
comparison_method = "ledoit_wolf"
comparison_df = compare_hrp_herc_weights(
    returns_df,
    covariance_method=comparison_method,
)
comparison_df.round(4)


In [ ]:
fig = plot_weight_comparison(
    comparison_df,
    title=f"HRP vs HERC Weights ({comparison_method})",
)
fig.show()


## Diversification Analysis

In [ ]:
def summarize_weights(weights: pd.Series, label: str) -> pd.Series:
    return pd.Series(
        {
            "Strategy": label,
            "Largest Weight": float(weights.max()),
            "Smallest Weight": float(weights.min()),
            "Weight Dispersion": float(weights.std()),
        }
    )

diversification_summary = pd.DataFrame(
    [
        summarize_weights(equal_weight, "Equal Weight"),
        summarize_weights(inverse_vol, "Inverse Volatility"),
        summarize_weights(hrp_weights_by_method[comparison_method], f"HRP ({comparison_method})"),
        summarize_weights(herc_weights_by_method[comparison_method], f"HERC ({comparison_method})"),
    ]
).set_index("Strategy")
diversification_summary.round(4)


## Discussion

In [ ]:
largest_difference_row = comparison_df.iloc[comparison_df["Difference"].abs().argmax()]
higher_weight_assets = comparison_df.loc[comparison_df["Difference"] > 0.0, "Asset"].tolist()

discussion_lines = [
    "### How does HERC differ from HRP?",
    "HERC traverses the actual linkage tree and equalizes risk between sibling branches using cluster volatility.",
    "HRP uses quasi-diagonal ordering plus midpoint recursive bisection with cluster variance-based scaling.",
    "",
    "### Which assets receive higher allocations?",
    f"Under `{comparison_method}`, HERC most strongly diverges on `{largest_difference_row['Asset']}`.",
    f"Assets with higher HERC weights than HRP: {', '.join(higher_weight_assets) if higher_weight_assets else 'None'}.",
    "",
    "### Why?",
    "Because HERC recomputes branch risk at each explicit tree split using local inverse-volatility cluster weights, capital migrates toward branches whose cluster volatility is lower under that local structure.",
]

display(Markdown("\n".join(discussion_lines)))
